In [4]:
from scipy.io import loadmat

data = loadmat("C:/Users/Minnie/Desktop/MCI/MCI1/1.mat")
print(data.keys())


dict_keys(['__header__', '__version__', '__globals__', 'export'])


In [ ]:
def load_all_mat_from_nested_folders(root_dir, label_value, window=512, stride=256):
    import numpy as np
    import os
    from scipy.io import loadmat

    X_list = []
    print(f"🔍 탐색 시작: {root_dir}")

    for subject in os.listdir(root_dir):
        subject_dir = os.path.join(root_dir, subject)
        if not os.path.isdir(subject_dir):
            continue

        for file in os.listdir(subject_dir):
            if not file.endswith(".mat"):
                continue

            path = os.path.join(subject_dir, file)
            try:
                mat = loadmat(path)
                if 'export' not in mat:
                    print(f"⚠️ 'export' key 없음: {path}")
                    continue

                eeg = mat['export']  # shape: (T, C) or (C, T)

                # 자동 transpose: 채널이 19가 아니면 전치
                if eeg.shape[0] != 19 and eeg.shape[1] == 19:
                    eeg = eeg.T

                ch, t = eeg.shape
                if t < window:
                    print(f"⚠️ 너무 짧은 EEG: {eeg.shape} in {path}")
                    continue

                for start in range(0, t - window + 1, stride):
                    epoch = eeg[:, start:start + window]
                    X_list.append(epoch)

                print(f"✅ 처리됨: {file}, epoch 수: {(t - window) // stride + 1}")

            except Exception as e:
                print(f"❌ 예외 발생: {file} - {e}")

    if len(X_list) == 0:
        raise ValueError(f"❌ 유효한 EEG가 없음: {root_dir}")

    X_arr = np.stack(X_list)
    y_arr = np.full(len(X_arr), label_value)
    print(f"✅ 총 epoch 수: {len(X_arr)} from {root_dir}")
    return X_arr, y_arr


In [35]:
def load_control_data_with_dense_stride(root_dir, label_value=2, window=512, stride=64):
    import numpy as np
    import os
    from scipy.io import loadmat

    X_list = []
    print(f"🚀 CONTROL 데이터 강화 시작: {root_dir} (stride={stride})")

    for subject in os.listdir(root_dir):
        subject_dir = os.path.join(root_dir, subject)
        if not os.path.isdir(subject_dir):
            continue

        for file in os.listdir(subject_dir):
            if not file.endswith(".mat"):
                continue

            path = os.path.join(subject_dir, file)
            try:
                mat = loadmat(path)
                if 'export' not in mat:
                    print(f"⚠️ 'export' key 없음: {path}")
                    continue

                eeg = mat['export']

                if eeg.shape[0] != 19 and eeg.shape[1] == 19:
                    eeg = eeg.T  # (채널, 시간)

                ch, t = eeg.shape
                if t < window:
                    continue

                for start in range(0, t - window + 1, stride):
                    epoch = eeg[:, start:start + window]
                    X_list.append(epoch)

            except Exception as e:
                print(f"❌ 파일 오류: {path} - {e}")

    if len(X_list) == 0:
        raise ValueError("❌ CONTROL 데이터를 불러오지 못했습니다.")

    X_arr = np.stack(X_list)
    y_arr = np.full(len(X_arr), label_value)
    print(f"✅ 생성된 CONTROL epoch 수: {len(X_arr)}")
    return X_arr, y_arr


In [36]:
X_ctrl, y_ctrl = load_control_data_with_dense_stride(
    "C:/Users/Minnie/Desktop/CONTROL", label_value=2, window=512, stride=64)


🚀 CONTROL 데이터 강화 시작: C:/Users/Minnie/Desktop/CONTROL (stride=64)
✅ 생성된 CONTROL epoch 수: 941


In [37]:
X_ad, y_ad = load_all_mat_from_nested_folders("C:/Users/Minnie/Desktop/AD", label_value=0)
X_mci, y_mci = load_all_mat_from_nested_folders("C:/Users/Minnie/Desktop/MCI", label_value=1)
X_ctrl, y_ctrl = load_all_mat_from_nested_folders("C:/Users/Minnie/Desktop/CONTROL", label_value=2)

X_all = np.concatenate([X_ad, X_mci, X_ctrl], axis=0)
y_all = np.concatenate([y_ad, y_mci, y_ctrl], axis=0)

print("✅ 전체 EEG shape:", X_all.shape)  # (전체 epoch 수, 19, 512)
print("✅ 라벨 분포:", np.unique(y_all, return_counts=True))



🔍 탐색 시작: C:/Users/Minnie/Desktop/AD
✅ 처리됨: 1.mat, epoch 수: 27
✅ 처리됨: 2.mat, epoch 수: 71
⚠️ 너무 짧은 EEG: (6776, 18) in C:/Users/Minnie/Desktop/AD\AD1\3.mat
✅ 처리됨: 4.mat, epoch 수: 92
✅ 처리됨: 5.mat, epoch 수: 121
✅ 처리됨: 1.mat, epoch 수: 13
✅ 처리됨: 2.mat, epoch 수: 9
✅ 처리됨: 3.mat, epoch 수: 16
✅ 처리됨: 4.mat, epoch 수: 20
✅ 처리됨: 5.mat, epoch 수: 14
✅ 처리됨: 6.mat, epoch 수: 12
✅ 처리됨: 7.mat, epoch 수: 25
✅ 처리됨: 8.mat, epoch 수: 94
✅ 처리됨: 9.mat, epoch 수: 9
✅ 처리됨: 1.mat, epoch 수: 147
✅ 처리됨: 2.mat, epoch 수: 84
✅ 처리됨: 3.mat, epoch 수: 61
✅ 처리됨: 4.mat, epoch 수: 98
✅ 처리됨: 1.mat, epoch 수: 44
✅ 처리됨: 2.mat, epoch 수: 36
✅ 처리됨: 3.mat, epoch 수: 45
✅ 처리됨: 4.mat, epoch 수: 147
✅ 처리됨: 5.mat, epoch 수: 20
✅ 처리됨: 6.mat, epoch 수: 58
✅ 처리됨: 7.mat, epoch 수: 26
✅ 처리됨: 8.mat, epoch 수: 20
✅ 처리됨: 1.mat, epoch 수: 148
✅ 처리됨: 2.mat, epoch 수: 147
✅ 처리됨: 3.mat, epoch 수: 50
✅ 처리됨: 1.mat, epoch 수: 21
✅ 처리됨: 2.mat, epoch 수: 32
✅ 처리됨: 3.mat, epoch 수: 55
✅ 처리됨: 4.mat, epoch 수: 68
✅ 처리됨: 5.mat, epoch 수: 43
✅ 처리됨: 6.mat, epoch 수: 64
✅ 처리됨: 7.mat

In [38]:
# (N, 19, 512) 기준
X_all = (X_all - X_all.mean(axis=(0, 2), keepdims=True)) / X_all.std(axis=(0, 2), keepdims=True)
X_all = np.nan_to_num(X_all)


In [65]:
import os
import numpy as np
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

def get_subject_files(root_dir):
    subjects = []
    for subject in os.listdir(root_dir):
        subject_path = os.path.join(root_dir, subject)
        if os.path.isdir(subject_path):
            files = [os.path.join(subject_path, f) for f in os.listdir(subject_path) if f.endswith(".mat")]
            if len(files) > 0:
                subjects.append(files)
    return subjects

def safe_split_subjects(subjects, test_ratio=0.2, val_ratio=0.15):
    if len(subjects) >= 3:
        train, test = train_test_split(subjects, test_size=test_ratio, random_state=42)
        train, val = train_test_split(train, test_size=val_ratio / (1 - test_ratio), random_state=42)
        return train, val, test
    elif len(subjects) == 2:
        train, test = train_test_split(subjects, test_size=0.5, random_state=42)
        return train, [], test
    else:
        return subjects, [], []

def slice_mat_files(subject_files, label_value, window=512, stride=256):
    X_list, y_list = [], []
    for file_list in subject_files:
        for file_path in file_list:
            try:
                mat = loadmat(file_path)
                if 'export' not in mat:
                    continue
                eeg = mat['export']
                if eeg.shape[0] != 19 and eeg.shape[1] == 19:
                    eeg = eeg.T
                ch, t = eeg.shape
                if t < window:
                    continue
                for start in range(0, t - window + 1, stride):
                    epoch = eeg[:, start:start + window]
                    X_list.append(epoch)
                    y_list.append(label_value)
            except Exception as e:
                print(f"❌ {file_path} 로드 실패: {e}")

    if len(X_list) == 0:
        print(f"⚠️ No valid EEG epochs found for label {label_value}")
        return np.empty((0, 19, window)), np.empty((0,), dtype=int)

    return np.stack(X_list), np.array(y_list)

# === Example usage ===
# 각 클래스별 subject list split
AD_subjects = get_subject_files("C:/Users/Minnie/Desktop/AD")
MCI_subjects = get_subject_files("C:/Users/Minnie/Desktop/MCI")
CTRL_subjects = get_subject_files("C:/Users/Minnie/Desktop/CONTROL")

train_AD, val_AD, test_AD = safe_split_subjects(AD_subjects)
train_MCI, val_MCI, test_MCI = safe_split_subjects(MCI_subjects)
train_CTRL, val_CTRL, test_CTRL = safe_split_subjects(CTRL_subjects)

# 슬라이싱 (CONTROL은 stride 더 작게)
X_train_AD, y_train_AD = slice_mat_files(train_AD, 0)
X_val_AD, y_val_AD     = slice_mat_files(val_AD, 0)
X_test_AD, y_test_AD   = slice_mat_files(test_AD, 0)

X_train_MCI, y_train_MCI = slice_mat_files(train_MCI, 1)
X_val_MCI, y_val_MCI     = slice_mat_files(val_MCI, 1)
X_test_MCI, y_test_MCI   = slice_mat_files(test_MCI, 1)

X_train_CTRL, y_train_CTRL = slice_mat_files(train_CTRL, 2, stride=64)
X_val_CTRL, y_val_CTRL     = slice_mat_files(val_CTRL, 2, stride=64)
X_test_CTRL, y_test_CTRL   = slice_mat_files(test_CTRL, 2, stride=64)

# 병합 (val은 CTRL 비어있으면 제외)
X_train = np.concatenate([X_train_AD, X_train_MCI, X_train_CTRL], axis=0)
y_train = np.concatenate([y_train_AD, y_train_MCI, y_train_CTRL], axis=0)

X_val = np.concatenate([X_val_AD, X_val_MCI], axis=0) if len(X_val_CTRL) == 0 else np.concatenate([X_val_AD, X_val_MCI, X_val_CTRL], axis=0)
y_val = np.concatenate([y_val_AD, y_val_MCI], axis=0) if len(y_val_CTRL) == 0 else np.concatenate([y_val_AD, y_val_MCI, y_val_CTRL], axis=0)

X_test = np.concatenate([X_test_AD, X_test_MCI, X_test_CTRL], axis=0)
y_test = np.concatenate([y_test_AD, y_test_MCI, y_test_CTRL], axis=0)

# 최종 shape 확인
print("✅ 데이터셋 준비 완료")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


⚠️ No valid EEG epochs found for label 2
✅ 데이터셋 준비 완료
Train: (8880, 19, 512) (8880,)
Val: (2977, 19, 512) (2977,)
Test: (3354, 19, 512) (3354,)


In [67]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # (1, 19, 512)
        return x, self.y[idx]

train_ds = EEGDataset(X_train, y_train)
val_ds = EEGDataset(X_val, y_val)
test_ds = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

In [68]:
import torch.nn as nn
import torch.nn.functional as F

class EEG_CNN(nn.Module):
    def __init__(self, num_classes=3):
        super(EEG_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(3, 5), padding=(1, 2))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(3, 5), padding=(1, 2))
        self.pool = nn.MaxPool2d((1, 2))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(32 * 19 * 128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (B, 16, 19, 256)
        x = self.pool(F.relu(self.conv2(x)))  # (B, 32, 19, 128)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [44]:
class EEG_RNN(nn.Module):
    def __init__(self, num_classes=3, hidden_size=64):
        super(EEG_RNN, self).__init__()
        self.rnn = nn.GRU(
            input_size=19,         # 채널 수
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # 입력: (B, 1, 19, 512) → (B, 512, 19)
        x = x.squeeze(1).permute(0, 2, 1)  # (B, Time, Channel)
        out, _ = self.rnn(x)               # (B, Time, Hidden*2)
        out = self.dropout(out[:, -1, :])  # 마지막 time step
        return self.fc(out)


In [71]:
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEG_CNN().to(device)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [72]:
for epoch in range(10):
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"📘 Epoch {epoch+1}, Loss: {running_loss:.4f}")


📘 Epoch 1, Loss: 143.3409
📘 Epoch 2, Loss: 92.5398
📘 Epoch 3, Loss: 66.5891
📘 Epoch 4, Loss: 57.3093
📘 Epoch 5, Loss: 50.4011
📘 Epoch 6, Loss: 40.4643
📘 Epoch 7, Loss: 35.0230
📘 Epoch 8, Loss: 29.6142
📘 Epoch 9, Loss: 25.6667
📘 Epoch 10, Loss: 22.8075


In [73]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch).argmax(dim=1).cpu().numpy()
        y_true.extend(y_batch.numpy())
        y_pred.extend(pred)

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=["AD", "MCI", "CONTROL"]))



📊 Classification Report:
              precision    recall  f1-score   support

          AD       0.85      0.99      0.91      2767
         MCI       0.23      0.04      0.07       504
     CONTROL       1.00      0.52      0.68        83

    accuracy                           0.83      3354
   macro avg       0.69      0.52      0.56      3354
weighted avg       0.76      0.83      0.78      3354



### 추가코드

# 1. 클래스 균형 맞추기
# 2. val / test evaluation
# 3. early stopping
# 4. 다른 cnn 모델

In [ ]:
# 1. 클래스 균형 맞추기

print(f"{X_all.shape=}")
print(f"{y_all.shape=}")


X_all.shape=(14511, 19, 512)
y_all.shape=(14511,)
